In [2]:
!pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.0-py2.py3-none-any.whl size=311317130 sha256=ed2edcbe0f892f12bec8b3a73cde85eda238c7f13591d1d4dc9cb3f6b985f17e
  Stored in directory: /root/.cache/pip/wheels/7b/1b/4b/3363a1d04368e7ff0d408e57ff57966fcdf00583774e761327
Successfully built pyspark


In [40]:
# Import the SparkSession class
from pyspark.sql import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    explode, floor, count, length, avg, format_number, col,
    split, from_json, asc,desc, size, array_intersect, collect_set, array,
    mean, max, min, when
)
from pyspark.rdd import RDD
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans


## Question 1

In [25]:

# Create a SparkSession object
spark = SparkSession.builder.appName("Inner Join").getOrCreate()

# Read the employee.csv file as a Spark DataFrame
df = spark.read.csv("employee.csv", header=True)

# Split the DataFrame into two DataFrames based on some condition
# For example, df1 contains employees with salary less than 5000
# and df2 contains employees with salary greater than or equal to 5000
df1 = df.filter(col("emp_salary") < 5000)
df2 = df.filter(col("emp_salary") >= 5000)

# Perform an inner join on emp_id between df1 and df2
# The result will be a DataFrame with columns from both df1 and df2
result = df1.join(df2, on="emp_id", how="inner")

# Show the result
result.show()

+------+--------+---------+----------+--------------+--------+--------+----------+--------------+
|emp_id|emp_name| emp_dept|emp_salary|emp_experience|emp_name|emp_dept|emp_salary|emp_experience|
+------+--------+---------+----------+--------------+--------+--------+----------+--------------+
|   102|     Bob|Marketing|    4500.0|           2.0| Charlie|      IT|    6000.0|           4.0|
|   109|    Iris|       HR|    4100.0|           1.5|   Henry|      IT|    6200.0|           4.5|
|   111|    Kate|     Code|    4900.0|           2.5|     Mia|      IT|    6100.0|           4.0|
+------+--------+---------+----------+--------------+--------+--------+----------+--------------+



## Question 2

In [26]:
result = df1.join(df2, on="emp_id", how="left")
# Show the result
result.show()

+------+--------+----------+----------+--------------+--------+--------+----------+--------------+
|emp_id|emp_name|  emp_dept|emp_salary|emp_experience|emp_name|emp_dept|emp_salary|emp_experience|
+------+--------+----------+----------+--------------+--------+--------+----------+--------------+
|   102|     Bob| Marketing|    4500.0|           2.0| Charlie|      IT|    6000.0|           4.0|
|   104|    Dave|        HR|    4000.0|           1.5|    null|    null|      null|          null|
|   106|   Frank|     Sales|    4800.0|           2.5|    null|    null|      null|          null|
|   107|    Gina| Marketing|    4700.0|           2.0|    null|    null|      null|          null|
|   109|    Iris|        HR|    4100.0|           1.5|   Henry|      IT|    6200.0|           4.5|
|   111|    Kate|      Code|    4900.0|           2.5|     Mia|      IT|    6100.0|           4.0|
|   112|     Lee|    Design|    4600.0|           2.0|    null|    null|      null|          null|
|   114|  

## Question 3

In [27]:
# Import the SparkSession class
from pyspark.sql import SparkSession

# Create a SparkSession object
spark = SparkSession.builder.appName("Total Salary").getOrCreate()

# Read the employee.csv file as a Spark DataFrame
df = spark.read.csv("employee.csv", header=True)

# Split the DataFrame into two DataFrames based on some condition
# For example, df1 contains employees with salary less than 5000
# and df2 contains employees with salary greater than or equal to 5000
a = df.filter(df.emp_salary < 5000)
b = df.filter(df.emp_salary >= 5000)

# Perform an inner join on emp_id between df1 and df2
# The result will be a DataFrame with columns like (emp_id, emp_name, emp_dept, emp_salary, emp_experience)
joined_df = a.join(b, on="emp_id", how="inner")

# Calculate the total salary of each employee by multiplying emp_salary with emp_experience
# The result will be a DataFrame with columns like (emp_id, emp_name, total_salary)
total_salary_df = joined_df.select("emp_id", (joined_df.emp_id * joined_df.emp_id).alias("total_salary"))

# Display the results using show()
total_salary_df.show()


+------+------------+
|emp_id|total_salary|
+------+------------+
|   102|     10404.0|
|   109|     11881.0|
|   111|     12321.0|
+------+------------+



## Question 4

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

# create SparkSession
spark = SparkSession.builder.appName("BroadcastJoin").getOrCreate()

# create the first RDD with key-value pairs
rdd1 = spark.sparkContext.parallelize([(1, "John"), (2, "Alice"), (3, "Bob")])

# create the first dataframe from the RDD
df1 = spark.createDataFrame(rdd1, ["id", "name"])

# create the second RDD with key-value pairs
rdd2 = spark.sparkContext.parallelize([(1, 25), (2, 30), (3, 35)])

# create the second dataframe from the RDD
df2 = spark.createDataFrame(rdd2, ["id", "age"])

# broadcast join
result = df1.join(broadcast(df2), ["id"], "inner")

# show the result
result.show()


+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| John| 25|
|  2|Alice| 30|
|  3|  Bob| 35|
+---+-----+---+



In [29]:
spark.stop()

## Question 5

In [30]:
from pyspark import SparkContext

# create a SparkContext object
sc = SparkContext("local", "Accumulator Example")

# create an RDD of integers from 1 to 10
RDD = sc.parallelize(range(1, 11))

# define an accumulator with an initial value of 0
accumulator = sc.accumulator(0)

# define a function to update the accumulator with each element
def add_to_accumulator(x):
    global accumulator
    accumulator += x

# use foreach() action on the RDD to update the accumulator with each element
RDD.foreach(add_to_accumulator)

# print the final value of the accumulator after processing all elements in the RDD
print("Sum of all values in RDD: ", accumulator.value)


Sum of all values in RDD:  55


In [55]:
sc.stop()

## Question 6

In [32]:
from pyspark.sql import SparkSession

# create a SparkSession object
spark = SparkSession.builder.appName("WineAnalysis").getOrCreate()

# read the wine dataset into a PySpark dataframe
wine_df = spark.read.csv("wine.csv", header=True, inferSchema=True)


In [33]:
# check the column names
print(wine_df.columns)

# check the data types of each column
print(wine_df.dtypes)

# check for missing values
from pyspark.sql.functions import isnan, when, count, col
wine_df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in wine_df.columns]).show()


['Alcohol', 'Malic_Acid', 'Ash', 'Ash_Alcanity', 'Magnesium', 'Total_Phenols', 'Flavanoids', 'Nonflavanoid_Phenols', 'Proanthocyanins', 'Color_Intensity', 'Hue', 'OD280', 'Proline']
[('Alcohol', 'double'), ('Malic_Acid', 'double'), ('Ash', 'double'), ('Ash_Alcanity', 'double'), ('Magnesium', 'int'), ('Total_Phenols', 'double'), ('Flavanoids', 'double'), ('Nonflavanoid_Phenols', 'double'), ('Proanthocyanins', 'double'), ('Color_Intensity', 'double'), ('Hue', 'double'), ('OD280', 'double'), ('Proline', 'int')]
+-------+----------+---+------------+---------+-------------+----------+--------------------+---------------+---------------+---+-----+-------+
|Alcohol|Malic_Acid|Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity|Hue|OD280|Proline|
+-------+----------+---+------------+---------+-------------+----------+--------------------+---------------+---------------+---+-----+-------+
|      0|         0|  0|           0|        0|       

In [34]:
wine_df.describe().show()


+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------------------+------------------+-----------------+-------------------+------------------+-----------------+
|summary|           Alcohol|        Malic_Acid|               Ash|     Ash_Alcanity|         Magnesium|     Total_Phenols|        Flavanoids|Nonflavanoid_Phenols|   Proanthocyanins|  Color_Intensity|                Hue|             OD280|          Proline|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------------------+------------------+-----------------+-------------------+------------------+-----------------+
|  count|               178|               178|               178|              178|               178|               178|               178|                 178|               178|              178|                178|          

In [35]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# create a VectorAssembler to combine all features into a single column
assembler = VectorAssembler(inputCols=wine_df.columns[:-1], outputCol="features")

# transform the data using the VectorAssembler
wine_features = assembler.transform(wine_df)

# use the Correlation function to calculate the correlation matrix
corr_matrix = Correlation.corr(wine_features, "features")

# convert the correlation matrix to a PySpark DataFrame
corr_df = corr_matrix.toPandas()

# display the correlation matrix
print(corr_df)


                                   pearson(features)
0  DenseMatrix([[ 1.        ,  0.09439694,  0.211...


## Question 7


In [41]:
wine_df = spark.read.csv("wine.csv", header=True, inferSchema=True)

# Rename the "class" column to "label"
wine_df = wine_df.withColumnRenamed("class", "label")

# Create a VectorAssembler to combine all features into a single column
assembler = VectorAssembler(inputCols=wine_df.columns[:-1], outputCol="features")

# Transform the data using the VectorAssembler
wine_features = assembler.transform(wine_df)

In [42]:
kmeans = KMeans(featuresCol="features", k=3)
model = kmeans.fit(wine_features)

# Get the cluster labels for each data point
predictions = model.transform(wine_features)


In [46]:
predictions.select("*").show()


+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+--------------------+----------+
|Alcohol|Malic_Acid| Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity| Hue|OD280|Proline|            features|prediction|
+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+--------------------+----------+
|  14.23|      1.71|2.43|        15.6|      127|          2.8|      3.06|                0.28|           2.29|           5.64|1.04| 3.92|   1065|[14.23,1.71,2.43,...|         1|
|   13.2|      1.78|2.14|        11.2|      100|         2.65|      2.76|                0.26|           1.28|           4.38|1.05|  3.4|   1050|[13.2,1.78,2.14,1...|         0|
|  13.16|      2.36|2.67|        18.6|      101|          2.8|      3.24|                 0.3|           2.81|